# TCML RL refiners SARL and MARL (leak-free)

Model 3 of the paper: the local refiner acting at the inter-branch cusps, trained on the corrected windows
(`data/rl_windows/rl_switch_windows_lf_*.npz`): centre of the window taken from the base prediction, the six
scalars computed on the true curve neutralised, target clamped to the action range [-1, 1].

In our own run (12 h, on CPU because the torch install had no network access) SARL completed its five seeds
and MARL diverged at epoch 100; see the README.


In [ ]:
import subprocess, sys
# the stock torch of the image does not support every GPU of the fleet;
# this needs 'Internet' enabled in the notebook settings
rc = subprocess.call([sys.executable, '-m', 'pip', 'install', '--quiet',
                      '--index-url', 'https://download.pytorch.org/whl/cu121', 'torch==2.4.1'])
print('pip install torch 2.4.1+cu121 exit code:', rc)
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())


In [ ]:
import os, sys, subprocess, shutil, time
from pathlib import Path
INPUT = Path('/kaggle/input')
# the dataset attached to this notebook must be this repository, data/ folder included
REPO_SRC = list(INPUT.rglob('train/train_star_pro.py'))[0].parents[1]
REPO = Path('/kaggle/working/TaylorCouetteML')
if REPO.exists(): shutil.rmtree(REPO)
shutil.copytree(REPO_SRC, REPO)
os.chdir(REPO)
print('repository ready at', REPO)


In [ ]:
OUT = Path('/kaggle/working/runs'); OUT.mkdir(parents=True, exist_ok=True)
EPOCHS, BATCH, LR, N_SEEDS = 1500, 16, 3e-4, 5
W = 'data/rl_windows/rl_switch_windows_lf_'
for script, out in [('train_sarl_v2.py', 'sarl_lf'), ('train_marl_v2.py', 'marl_lf')]:
    args = [sys.executable, 'train/' + script, '--windows_train', W + 'train.npz',
            '--windows_val', W + 'val.npz', '--epochs', str(EPOCHS), '--batch', str(BATCH),
            '--lr', str(LR), '--n_seeds', str(N_SEEDS), '--out_dir', str(OUT / out)]
    print('>>>', ' '.join(args)); t0 = time.time()
    rc = subprocess.call(args, cwd=str(REPO))
    print('<<<', script, 'exit', rc, 'elapsed', round((time.time() - t0) / 60, 1), 'min')


In [ ]:
for root, dirs, files in os.walk(OUT):
    for f in sorted(files):
        p = Path(root) / f
        print(p.relative_to(OUT), round(p.stat().st_size / 1e6, 1), 'MB')
